# Module 4 Homework: Analytics Engineering with dbt

### Question 1. dbt Lineage and Execution

Given a dbt project with the following structure:

Given a dbt project with the following structure:

```
models/
├── staging/
│   ├── stg_green_tripdata.sql
│   └── stg_yellow_tripdata.sql
└── intermediate/
    └── int_trips_unioned.sql (depends on stg_green_tripdata & stg_yellow_tripdata)
```

If you run `dbt run --select int_trips_unioned`, what models will be built?

- `stg_green_tripdata`, `stg_yellow_tripdata`, and `int_trips_unioned` (upstream dependencies)
- Any model with upstream and downstream dependencies to `int_trips_unioned`
- `int_trips_unioned` only
- `int_trips_unioned`, `int_trips`, and `fct_trips` (downstream dependencies)

Simple English Logic

The Command: dbt run --select int_trips_unioned

The Rule: In dbt, if you select a specific model name without any plus signs (+), dbt will run only that specific model.

* Upstream vs. Downstream:

  - To run Upstream (parents) -> use +int_trips_unioned.

  - To run Downstream (children) -> use int_trips_unioned+.

  - To run Only the model -> use the name.

**Answer:** int_trips_unioned only.

---

### Question 2. dbt Tests

You've configured a generic test like this in your `schema.yml`:

```yaml
columns:
  - name: payment_type
    data_tests:
      - accepted_values:
          arguments:
            values: [1, 2, 3, 4, 5]
            quote: false
```

Your model `fct_trips` has been running successfully for months. A new value `6` now appears in the source data.

What happens when you run `dbt test --select fct_trips`?

- dbt will skip the test because the model didn't change
- dbt will fail the test, returning a non-zero exit code
- dbt will pass the test with a warning about the new value
- dbt will update the configuration to include the new value

In dbt, Generic Tests (like accepted_values) are queries that run against your existing data in the database.

- The Rule: You told dbt that the payment_type column is only allowed to have values 1, 2, 3, 4, or 5.
- The Violation: When a new value 6 appears in the source data, it breaks this rule.
- The Result: When you run dbt test, dbt executes a SQL query to find any values that are not in your list. Since it finds the value 6, the test fails.
**Answer: dbt will fail the test, returning a non-zero exit code**

---

### Question 3. Counting Records in `fct_monthly_zone_revenue`

After running your dbt project, query the `fct_monthly_zone_revenue` model.

What is the count of records in the `fct_monthly_zone_revenue` model?

- 12,998
- 14,120
- 12,184
- 15,421

In [ ]:
1. The Dependency Chain
Think of it as a "Recipe." If the ingredients (Staging tables) are missing 2019 data, the final cake (Monthly Revenue table) will be too small. You need to run the entire family tree.

2. The Incremental Trap
dbt is "lazy" by default. It might only look for new data and skip the old 2019 records. This is why you originally saw only 4,000+ rows instead of 12,000+.

3. The "Plus" (+) and "Full Refresh" Solution

--full-refresh: Tells dbt to "Clear the table and start from zero."

+ prefix: Tells dbt to "Refresh all the parent models (upstream) first."
This ensures every single trip from 2019 and 2020 is counted.

Or you can just use math: 24 months × 2 taxi types × ~260 zones -> get around 12,998 rows.

**Answer: 12,998**

---

### Question 4. Best Performing Zone for Green Taxis (2020)

Using the `fct_monthly_zone_revenue` table, find the pickup zone with the **highest total revenue** (`revenue_monthly_total_amount`) for **Green** taxi trips in 2020.

Which zone had the highest revenue?

- East Harlem North
- Morningside Heights
- East Harlem South
- Washington Heights South


1. The table has data from 2019 to 2021. You must filter for Year = 2020 and Service = Green.
Since the column is a date (revenue_month), we use EXTRACT(YEAR FROM ...).

2. The Aggregation
The table shows monthly revenue. To find the "Yearly Winner," you must SUM all the monthly amounts for each pickup_zone.

3. The Sorting
Order the results by total revenue in Descending order (Highest to Lowest). The top row is your answer.

```sql
SELECT 
    pickup_zone,
    SUM(revenue_monthly_total_amount) AS total_revenue
FROM `de-zoomcamp-487123.trips_data_all.fct_monthly_zone_revenue`
WHERE EXTRACT(YEAR FROM revenue_month) = 2020 
  AND service_type = 'Green'
GROUP BY 1
```
**Answer:** East Harlem North

---

### Question 5. Green Taxi Trip Counts (October 2019)

Using the `fct_monthly_zone_revenue` table, what is the **total number of trips** (`total_monthly_trips`) for Green taxis in October 2019?

- 500,234
- 350,891
- 384,624
- 421,509


1. The Filter
Narrow down the huge dataset to only three conditions:

Year: 2019

Month: 10 (October)

Service Type: Green

2. The SUM
Because the table is broken down by zones, you must SUM the total_monthly_trips column to get the total count for the entire city.

```sql
SELECT 
    SUM(total_monthly_trips) AS total_trips
FROM `de-zoomcamp-487123.trips_data_all.fct_monthly_zone_revenue`

WHERE EXTRACT(YEAR FROM revenue_month) = 2019 
  AND EXTRACT(MONTH FROM revenue_month) = 10
  AND service_type = 'Green';

**Answer:** 421,509

---

### Question 6. Build a Staging Model for FHV Data

Create a staging model for the **For-Hire Vehicle (FHV)** trip data for 2019.

1. Load the [FHV trip data for 2019](https://github.com/DataTalksClub/nyc-tlc-data/releases/tag/fhv) into your data warehouse
2. Create a staging model `stg_fhv_tripdata` with these requirements:
   - Filter out records where `dispatching_base_num IS NULL`
   - Rename fields to match your project's naming conventions (e.g., `PUlocationID` → `pickup_location_id`)

What is the count of records in `stg_fhv_tripdata`?

- 42,084,899
- 43,244,693
- 22,998,722
- 44,112,187

Step 1 Data Loading: Load the raw 2019 FHV data into BigQuery.

Step 2 Configuration: Update your schema.yml file to tell dbt where the raw fhv_tripdata table is located.

Step 3:Create a file named stg_fhv_tripdata.sql. Inside this file, you do two things:

  * Rename Columns: Change names like PUlocationID to pickup_location_id.

  * Filter (The WHERE Clause): Add WHERE dispatching_base_num IS NOT NULL.

Step 4 The Build:
Run the command: dbt build --select stg_fhv_tripdata.
dbt will read the millions of raw rows, apply your filter, and save only the clean rows into a new table.

```sql
SELECT count(*) 
FROM `de-zoomcamp-487123.trips_data_all.stg_fhv_tripdata`;
```

**Answer:** 43,244,693